In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import json
from typing import Dict, List

import boto3
import botocore
import requests


S3_BUCKET = "cubix-chicago-taxi-bb-v2"
TAXI_API_URL = "https://data.cityofchicago.org/resource/ajtu-isnz.json"
WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/era5"

s3_client = boto3.client("s3")


def _get_data_from_api(url: str, params: Dict = None) -> List[Dict]:
    """
    Retrieves data from the given api with optional parameters.

    :param url:     The URL to retrieve the data from.
    :param params:  Optinally send parameters with the request.
    :return:        A list of dictionaries containing the data.
    """
    print(f"Retrieving data from {url}")
    response = requests.get(url, params=params)
    return response.json()


def get_taxi_data(date_str: str) -> List[Dict]:
    """
    Retrieves taxi data from the given date.

    :param date_str:    The date to retrieve the data from.
    :return:            A list of dictionaries containing the taxi data.
    """
    print(f"Retrieving taxi data from {date_str}")
    query = f"?$where=trip_start_timestamp >= '{date_str}T00:00:00' AND trip_start_timestamp <= '{date_str}T23:59:59'&$limit=50000"
    return _get_data_from_api(TAXI_API_URL + query)


def get_weather_data(date_str: str) -> List[Dict]:
    """
    Retrieves weather data from the given date.

    :param date_str:    The date to retrieve the data from.
    :return:            A list of dictionaries containing the weather data.
    """
    print(f"Retrieving weather data from {date_str}")
    params = {
        "latitude": 41.85,
        "longitude": -87.65,
        "start_date": date_str,
        "end_date": date_str,
        "hourly": "temperature_2m,wind_speed_10m,rain,precipitation"
    }
    return _get_data_from_api(WEATHER_API_URL, params)


def upload_to_s3(data: Dict, folder: str, filename: str) -> None:
    """
    Uploads the given data to S3.

    :param data:            The data to upload.
    :param folder:          The folder to upload the data to.
    :param filename:        The filename to upload the data to.
    :raise ValueError:      If "data" is None or empty.
    :raise RuntimeError:    If there is an error uploading to S3.
    """
    print(f"Uploading {filename} to S3.")

    if not data:
        raise ValueError(f"Data for {filename} is empty! Stopping execution.")

    s3_key = f"raw_data/to_processed/{folder}/{filename}"

    try:
        s3_client.put_object(
            Body=json.dumps(data),
            Bucket=S3_BUCKET,
            Key=s3_key
        )
        print(f"Uploaded {filename} to S3.")
    except Expection as e:
        raise RuntimeError(f"Error uploading {filename} to S3: {e}") from e    


def lambda_handler(event, context):
    """
    Lambda function to retrieve taxi and weather data from the given date and upload it to S3.

    Steps:
        1. Create the date which is today minus 2 months.
        2. Get the taxi raw data.
        3. Get the weather raw data.
        4. Upload them to S3.
    """
    date_str = (datetime.now() - relativedelta(months=2)).strftime("%Y-%m-%d")

    taxi_data = get_taxi_data(date_str)
    weather_data = get_weather_data(date_str)

    upload_to_s3(taxi_data, "taxi", f"taxi_{date_str}.json")
    upload_to_s3(weather_data, "weather", f"weather_{date_str}.json")
